# Generate the FD002 and FD003 representations and initial detector outputs.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,platform,time,gc,importlib.metadata
import numpy as np
import pandas as pd
import sklearn,joblib
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.metrics import (average_precision_score,roc_auc_score,precision_recall_curve,
                            precision_score,recall_score,f1_score,confusion_matrix)


DATA_DIR=Path('data')
PROTOCOL_DIR=Path('revision_outputs')/'09_FD002_FD003_protocol'
OUTPUT_ROOT=Path('revision_outputs')/'10_FD002_FD003_grouped_cv'


In [ ]:
try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError("Bu notebook TensorFlow gerektiriyor. TensorFlow kurulu kernel'i seçin; yoksa bu hata mesajını paylaşın.") from exc
print("TensorFlow:",tf.__version__)
print("Python:",platform.python_version(),"| scikit-learn:",sklearn.__version__)


In [ ]:
def require(ok, message):
    if not bool(ok): raise ValueError(message)

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def choose_threshold(y, scores):
    require(np.isfinite(scores).all() and len(np.unique(y))==2, "Geçersiz seçim skorları/etiketleri.")
    p,r,t=precision_recall_curve(y, scores)
    f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(t),where=(p[:-1]+r[:-1])>0)

    i=np.flatnonzero(np.isclose(f,f.max(),rtol=0,atol=1e-12))[-1]
    return float(t[i])

def fit_candidate(method, params, X, seed):
    if method=="IF":
        model=IsolationForest(**params,max_samples="auto",contamination="auto",random_state=seed,n_jobs=-1)
    elif method=="LOF":
        require(params["n_neighbors"]<len(X), "LOF için yetersiz fit örneği.")
        model=LocalOutlierFactor(**params,novelty=True,contamination="auto",metric="euclidean",n_jobs=-1)
    else:
        require(params["n_neighbors"]<=len(X), "kNN için yetersiz fit örneği.")
        model=NearestNeighbors(**params,metric="euclidean",n_jobs=-1)
    return model.fit(X)

def score(method, model, X):


    values = (model.kneighbors(X,return_distance=True)[0][:,-1]
              if method=="kNN" else -model.score_samples(X))
    require(np.isfinite(values).all(), "Sonlu olmayan model skoru.")
    return values

def metrics(y,scores,threshold):
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"AP":float(average_precision_score(y,scores)),
            "ROC_AUC":float(roc_auc_score(y,scores)),
            "precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "F1":float(f1_score(y,pred,zero_division=0)),
            "Macro_F1":float(f1_score(y,pred,average="macro",zero_division=0)),
            "TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn)}

def canonical(value):
    return json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False)

def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    temp.replace(path)

def scaler_parameters(s):
    return {'mean':s.mean_.tolist(),'scale':s.scale_.tolist(),'variance':s.var_.tolist()}


In [ ]:
APPROVED_SUMMARY = load_json('assets/10/04_approved_summary.json')
EXPECTED_PROTOCOL='39d18d13a674a59df953fbaf7ffdc0a20adb198f41bb1a55a4f525553128dc8a'
EXPECTED_VERSIONS={'python': '3.12.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'tensorflow': '2.21.0'}
IMPLEMENTATION_ID="FD002_FD003_grouped_cv_v1"
SENSORS=[f"Sensor{i}" for i in range(1,22)]
SETTINGS=["OpSet1","OpSet2","OpSet3"]
COLUMNS=["UnitNumber","Time"]+SETTINGS+SENSORS


In [ ]:
def validate_metadata(m,ds,rows,units):
    cols=['dataset','split','source_row','UnitNumber','Time','engine_key','sample_key','RUL']
    require(set(cols).issubset(m.columns) and len(m)==rows,'Eğitim metadata şeması/satır sayısı farklı.')
    require(m[cols].notna().all().all(),'Metadata içinde eksik değer var.')
    require(m.dataset.eq(ds).all() and m.split.eq('train').all(),'Veri kümesi/bölme karışmış.')
    m=m.copy()
    for col in ['source_row','UnitNumber','Time','RUL']:
        a=pd.to_numeric(m[col],errors='raise').to_numpy(dtype=float)
        require(np.isfinite(a).all() and np.all(a==np.floor(a)),f'{col} sonlu tam sayı olmalı.')
        m[col]=a.astype('int64')
    require(np.array_equal(m.source_row.to_numpy(),np.arange(rows)),'source_row sırası değişmiş.')
    require(np.array_equal(np.sort(m.UnitNumber.unique()),np.arange(1,units+1)),'Motor kimlikleri farklı.')
    require(not m.duplicated(['UnitNumber','Time']).any(),'Tekrarlanan motor/çevrim var.')
    engines=ds+':train:'+m.UnitNumber.astype(str)
    require(m.engine_key.eq(engines).all(),'Motor anahtarı geçersiz.')
    require(m.sample_key.eq(engines+':'+m.Time.astype(str)).all() and m.sample_key.is_unique,'Örnek anahtarı geçersiz.')
    for engine,g in m.groupby('engine_key',sort=False):
        require(np.array_equal(g.Time.to_numpy(),np.arange(1,len(g)+1)),f'{engine}: çevrim sırası bozuk.')
        require(np.array_equal(g.RUL.to_numpy(),g.Time.max()-g.Time.to_numpy()),f'{engine}: RUL hesabı farklı.')
    return m

def validate_splits(s,ds,keys):
    keys=set(keys);n=len(keys);size=n//10
    require(s.dataset.eq(ds).all() and set(s.scheme)=={'cv','final'},'Bölme kimliği farklı.')
    require(not s.duplicated(['scheme','fold','engine_key']).any(),'Aynı motor birden fazla role atanmış.')
    cv=s.loc[s.scheme.eq('cv')]
    require(set(cv.fold)==set(range(1,11)),'Dış fold kapsamı farklı.')
    for _,g in cv.groupby('fold'):
        require(len(g)==n and set(g.engine_key)==keys,'Bir fold tüm motorları tam bir kez içermeli.')
        require(g.role.value_counts().to_dict()=={'fit':7*size,'early_stop':size,'selection':size,'outer_valid':size},'CV rol sayıları farklı.')
    outer=cv.loc[cv.role.eq('outer_valid'),'engine_key'].value_counts()
    require(set(outer.index)==keys and outer.eq(1).all(),'Her motor tam bir kez dış değerlendirmede olmalı.')
    final=s.loc[s.scheme.eq('final')]
    require(len(final)==n and set(final.engine_key)==keys and final.fold.eq(0).all(),'Nihai bölme kapsamı farklı.')
    require(final.role.value_counts().to_dict()=={'fit':8*size,'early_stop':size,'selection':size},'Nihai rol sayıları farklı.')

def load_inputs(data_dir,protocol_dir,approved):
    summary_path=protocol_dir/'additional_protocol_summary.json'
    require(summary_path.is_file(),f'09 özeti bulunamadı: {summary_path}')
    s=json.loads(summary_path.read_text(encoding='utf-8'))
    require(s==approved,'09 özeti onaylanan çıktıdan farklı.')
    for relative,h in approved['output_file_sha256'].items():
        path=protocol_dir/relative
        require(path.is_file() and digest(path)==h,f'09 çıktısı eksik/değişmiş: {path}')
    p=json.loads((protocol_dir/'experiment_protocol.json').read_text(encoding='utf-8'))
    require(hashlib.sha256(canonical(p).encode()).hexdigest()==approved['canonical_protocol_sha256'],
            'Protokol içerik hash değeri farklı.')
    require(p['source_training_files']==approved['source_training_files'],'Kaynak veri planı farklı.')
    splits=pd.read_csv(protocol_dir/'engine_splits.csv')
    raw={};refs={}
    for ds,info in approved['source_training_files'].items():
        path=data_dir/f'train_{ds}.txt'
        require(path.is_file() and digest(path)==info['train_raw_sha256'],f'Ham eğitim dosyası eksik/değişmiş: {path}')
        frame=pd.read_csv(path,sep=r'\s+',header=None)
        require(frame.shape==(info['rows'],26),'Ham eğitim boyutu farklı.')
        frame.columns=COLUMNS
        require(np.isfinite(frame.to_numpy(dtype=float)).all(),'Sonlu olmayan ham değer.')
        ref=pd.read_csv(protocol_dir/ds/'train_evaluation_reference.csv')
        ref=validate_metadata(ref,ds,info['rows'],info['units'])
        require(np.array_equal(frame[['UnitNumber','Time']].to_numpy(),ref[['UnitNumber','Time']].to_numpy()),
                'Ham sensör satırları ile referans motor/çevrim sırası uyuşmuyor.')
        for w in [20,30,40]:
            require(np.array_equal(ref[f'late_life_le_{w}'].to_numpy(),ref.RUL.le(w).astype(int).to_numpy()),
                    'RUL referansı değişmiş.')
        validate_splits(splits.loc[splits.dataset.eq(ds)],ds,ref.engine_key.unique())
        raw[ds]=frame;refs[ds]=ref
    return p,splits,raw,refs

protocol,splits,raw_data,references=load_inputs(DATA_DIR,PROTOCOL_DIR,APPROVED_SUMMARY)
require(APPROVED_SUMMARY['status']=='PASS' and EXPECTED_PROTOCOL==APPROVED_SUMMARY['canonical_protocol_sha256'],
        'Onaylanan protokol kimliği farklı.')
VERSIONS={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
          'sklearn':sklearn.__version__,'tensorflow':tf.__version__}
for key,value in EXPECTED_VERSIONS.items():
    require(VERSIONS[key]==value,f'{key}={VERSIONS[key]}, beklenen={value}. Python (C-MAPSS Revision) kernelini seçin.')
VERSIONS.update(keras=tf.keras.__version__,scipy=importlib.metadata.version('scipy'),joblib=joblib.__version__)
BLOCKS=protocol['analysis_blocks'];SEEDS=tuple(protocol['model_seeds']);FOLDS=tuple(range(1,11))
REPRESENTATIONS=protocol['representations']
CANDIDATES={k:v['candidates'] for k,v in protocol['detectors'].items()}
run_plan={'stage':'10_additional_grouped_cv','implementation_id':IMPLEMENTATION_ID,
          'protocol_sha256':EXPECTED_PROTOCOL,'protocol':protocol,'versions':VERSIONS,
          'platform':platform.platform(),'machine':platform.machine(),
          'test_files_read':False,'completed_unit':'dataset/preprocessing/seed/fold/representation'}
PLAN_SHA256=hashlib.sha256(canonical(run_plan).encode()).hexdigest()
RUN_DIR=OUTPUT_ROOT/PLAN_SHA256[:20];RUN_DIR.mkdir(parents=True,exist_ok=True)
plan_path=RUN_DIR/'run_plan.json'
if plan_path.exists():require(json.loads(plan_path.read_text(encoding='utf-8'))==run_plan,'Kayıtlı çalışma planı farklı.')
else:write_json(plan_path,run_plan)
print('09 protokolü ve eğitim dosyaları doğrulandı. Test dosyası okunmadı.')
print('Çıktı klasörü:',RUN_DIR.resolve())


In [ ]:
def fit_preprocessor(frame,fit_mask,mode):
    X=frame[SENSORS].to_numpy(dtype=np.float32)
    require(fit_mask.dtype==bool and fit_mask.shape==(len(frame),),'Geçersiz fit maskesi.')
    if mode=='global':
        scaler=StandardScaler().fit(X[fit_mask])
        return {'mode':mode,'sensor_scaler':scaler}
    require(mode=='conditionwise','Tanımsız ön işleme.')
    cfg=protocol['preprocessing']['conditionwise']
    settings=frame[SETTINGS].to_numpy(dtype=np.float64)
    settings_scaler=StandardScaler().fit(settings[fit_mask])
    fit_settings=settings_scaler.transform(settings[fit_mask])
    km=KMeans(**cfg['KMeans']).fit(fit_settings)
    labels=km.predict(fit_settings)
    counts=np.bincount(labels,minlength=6)
    require(len(counts)==6 and np.all(counts>=cfg['minimum_fit_rows_per_cluster']),
            'Altı çalışma koşulunun her birinde en az iki fit satırı olmalı; yedek dönüşüm uygulanmadı.')
    sensor_scalers={k:StandardScaler().fit(X[fit_mask][labels==k]) for k in range(6)}
    return {'mode':mode,'settings_scaler':settings_scaler,'kmeans':km,'sensor_scalers':sensor_scalers,
            'fit_cluster_counts':counts.tolist()}

def transform_preprocessor(pre,frame):
    X=frame[SENSORS].to_numpy(dtype=np.float32)
    if pre['mode']=='global':
        out=pre['sensor_scaler'].transform(X).astype(np.float32,copy=False)
        labels=None
    else:
        settings=pre['settings_scaler'].transform(frame[SETTINGS].to_numpy(dtype=np.float64))
        labels=pre['kmeans'].predict(settings)
        out=np.empty_like(X)
        for k,scaler in pre['sensor_scalers'].items():
            mask=labels==k
            if mask.any():out[mask]=scaler.transform(X[mask])
    require(out.shape==(len(frame),21) and np.isfinite(out).all(),'Geçersiz sensör dönüşümü.')
    return out,labels

def describe_preprocessor(pre):
    if pre['mode']=='global':return {'mode':'global','sensor_scaler':scaler_parameters(pre['sensor_scaler'])}
    return {'mode':'conditionwise','fit_cluster_counts':pre['fit_cluster_counts'],
            'settings_scaler':scaler_parameters(pre['settings_scaler']),
            'centers_scaled_settings':pre['kmeans'].cluster_centers_.tolist(),
            'centers_original_settings':pre['settings_scaler'].inverse_transform(pre['kmeans'].cluster_centers_).tolist(),
            'sensor_scalers':{str(k):scaler_parameters(s) for k,s in pre['sensor_scalers'].items()}}

def prepare_fold(ds,mode,fold_id):
    require({'dataset':ds,'preprocessing':mode} in BLOCKS,'Plan dışı veri/ön işleme bloğu.')
    ref=references[ds];frame=raw_data[ds]
    subset=splits.loc[splits.dataset.eq(ds)&splits.scheme.eq('cv')&splits.fold.eq(fold_id)]
    expected=APPROVED_SUMMARY['datasets'][ds]['cv_engines_per_role']
    require(subset.engine_key.is_unique and subset.role.value_counts().to_dict()==expected,'Motor rolleri farklı.')
    roles=ref.engine_key.map(subset.set_index('engine_key').role)
    require(roles.notna().all(),'Atanmamış motor.')
    masks={role:roles.eq(role).to_numpy() for role in ['fit','early_stop','selection','outer_valid']}
    pre=fit_preprocessor(frame,masks['fit'],mode)
    Xs={};cluster_counts={}
    for role,mask in masks.items():
        Xs[role],labels=transform_preprocessor(pre,frame.loc[mask])
        if labels is not None:cluster_counts[role]=np.bincount(labels,minlength=6).tolist()
    y=ref.loc[masks['selection'],'late_life_le_30'].to_numpy(dtype=int)
    require(len(np.unique(y))==2,'Selection referansı tek sınıf.')
    info={'dataset':ds,'preprocessing':mode,'fold':fold_id,
          'roles':{role:{'engines':int(ref.loc[m,'engine_key'].nunique()),'rows':int(m.sum())} for role,m in masks.items()},
          'preprocessor':describe_preprocessor(pre),'cluster_rows_by_role':cluster_counts}
    return masks,pre,Xs,y,info


In [ ]:
def build_ae(config,seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    d=config["latent_dim"]
    inputs=tf.keras.Input(shape=(21,),name="sensor_input")
    h=tf.keras.layers.Dense(42,activation="relu",name="encoder_42")(inputs)
    if config["architecture"]=="deep":
        h=tf.keras.layers.Dense(21,activation="relu",name="encoder_21")(h)
    latent=tf.keras.layers.Dense(d,activation="linear",name="latent")(h)
    h=latent
    if config["architecture"]=="deep":
        h=tf.keras.layers.Dense(21,activation="relu",name="decoder_21")(h)
    h=tf.keras.layers.Dense(42,activation="relu",name="decoder_42")(h)
    outputs=tf.keras.layers.Dense(21,activation="linear",name="reconstruction")(h)
    ae=tf.keras.Model(inputs,outputs,name=config["name"])
    encoder=tf.keras.Model(inputs,latent,name=config["name"]+"_encoder")
    ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss="mse")
    return ae,encoder

def fit_representation(config,seed,Xs,OUT):
    ae,encoder=build_ae(config,seed)
    early=tf.keras.callbacks.EarlyStopping(monitor="val_loss",mode="min",patience=10,restore_best_weights=True)
    started=time.perf_counter()
    history=ae.fit(Xs["fit"],Xs["fit"],validation_data=(Xs["early_stop"],Xs["early_stop"]),
                   epochs=100,batch_size=256,shuffle=True,callbacks=[early],verbose=0)
    elapsed=time.perf_counter()-started
    require(np.isfinite(history.history["loss"]).all() and np.isfinite(history.history["val_loss"]).all(),
            "AE eğitiminde sonlu olmayan loss.")
    ae.save(OUT/"autoencoder.keras")
    encoder.save(OUT/"encoder.keras")
    pd.DataFrame(history.history).assign(epoch=np.arange(1,len(history.history["loss"])+1)).to_csv(OUT/"ae_history.csv",index=False)
    Z={role:encoder.predict(Xs[role],batch_size=256,verbose=0) for role in ["fit","selection","outer_valid"]}
    for role,z in Z.items():
        require(z.shape==(len(Xs[role]),config["latent_dim"]) and np.isfinite(z).all(),"Geçersiz latent çıktı.")
    latent_scaler=StandardScaler().fit(Z["fit"])
    Zs={role:latent_scaler.transform(z) for role,z in Z.items()}
    ae_scores={}
    for role in ["selection","outer_valid"]:
        reconstructed=ae.predict(Xs[role],batch_size=256,verbose=0)
        ae_scores[role]=np.mean(np.square(Xs[role]-reconstructed),axis=1)
        require(np.isfinite(ae_scores[role]).all(),"Geçersiz reconstruction skoru.")
    ae_info={"epochs_run":len(history.history["loss"]),
             "best_epoch":int(np.argmin(history.history["val_loss"])+1),
             "best_early_stop_loss":float(min(history.history["val_loss"])),
             "final_training_loss":float(history.history["loss"][-1]),
             "final_early_stop_loss":float(history.history["val_loss"][-1]),
             "parameter_count":int(ae.count_params()),"training_seconds":elapsed,
             "latent_fit_std":Z["fit"].std(axis=0).tolist()}
    del ae,encoder
    return Zs,ae_scores,latent_scaler,ae_info

def representation_arrays(name,seed,Xs,out):
    if name=='Raw':return {r:Xs[r] for r in ['fit','selection','outer_valid']},None,{'name':name}
    if name in ['PCA_d4','PCA_scaled_d4']:
        pca=PCA(n_components=4,svd_solver='full',whiten=False).fit(np.asarray(Xs['fit'],dtype=np.float64))
        Z={r:pca.transform(np.asarray(Xs[r],dtype=np.float64)) for r in ['fit','selection','outer_valid']}
        post=None
        if name=='PCA_scaled_d4':
            post=StandardScaler().fit(Z['fit']);Z={r:post.transform(z) for r,z in Z.items()}
        joblib.dump({'pca':pca,'post_scaler':post},out/'representation.joblib',compress=3)
        info={'name':name,'explained_variance_ratio':pca.explained_variance_ratio_.tolist(),
              'cumulative_explained_variance_ratio':float(pca.explained_variance_ratio_.sum()),
              'post_scaler':None if post is None else scaler_parameters(post)}
        return Z,None,info
    require(name in ['AE_deep_d21','AE_deep_d4'],'Plan dışı AE.')
    config={'name':name,'architecture':'deep','latent_dim':21 if name=='AE_deep_d21' else 4}
    Z,scores,latent_scaler,info=fit_representation(config,seed,Xs,out)
    joblib.dump(latent_scaler,out/'latent_scaler.joblib',compress=3)
    info.update(name=name,latent_dim=config['latent_dim'],latent_scaler=scaler_parameters(latent_scaler))
    tf.keras.backend.clear_session();gc.collect()
    return Z,scores,info


In [ ]:
def subjob_path(ds,mode,seed,fold,name):
    return RUN_DIR/ds/mode/f'seed_{seed}'/f'fold_{fold:02d}'/name

def cached_job(ds,mode,seed,fold,name):
    out=subjob_path(ds,mode,seed,fold,name);marker=out/'result.json'
    if not marker.is_file():return None
    result=json.loads(marker.read_text(encoding='utf-8'))
    identity={'dataset':ds,'preprocessing':mode,'seed':seed,'fold':fold,'representation':name}
    require(result['identity']==identity and result['plan_sha256']==PLAN_SHA256,'Kayıtlı iş kimliği farklı.')
    methods=set(CANDIDATES)|({'MSE'} if name.startswith('AE_') else set())
    require(result['status']=='PASS' and len(result['metrics'])==len(methods)*3,'Kayıtlı iş eksik.')
    require({(r['method'],r['RUL_window']) for r in result['metrics']}=={(m,w) for m in methods for w in [20,30,40]},'Kayıtlı metrik kapsamı farklı.')
    for r in result['metrics']:
        require(all(r[k]==v for k,v in identity.items()),'Kayıtlı metrik kimliği farklı.')
    require(result['output_sha256'],'Kayıtlı işte dosya hash listesi boş.')
    for relative,h in result['output_sha256'].items():
        path=out/relative
        require(path.is_file() and digest(path)==h,f'Tamamlanmış çıktı eksik/değişmiş: {path}')
    return result

def run_subjob(ds,mode,seed,fold,name,prepared):
    cached=cached_job(ds,mode,seed,fold,name)
    if cached is not None:return cached
    masks,pre,Xs,y_selection,fold_info=prepared
    out=subjob_path(ds,mode,seed,fold,name);out.mkdir(parents=True,exist_ok=True)
    identity={'dataset':ds,'preprocessing':mode,'seed':seed,'fold':fold,'representation':name}
    started=time.perf_counter()
    Z,mse_scores,representation_info=representation_arrays(name,seed,Xs,out)
    require(all(np.isfinite(z).all() for z in Z.values()),'Sonlu olmayan temsil.')
    joblib.dump(pre,out/'preprocessor.joblib',compress=3)
    write_json(out/'preprocessing_parameters.json',fold_info)
    chosen={};trials=[]
    for method,candidates in CANDIDATES.items():
        print('  Dedektör:',method,flush=True)
        best=None
        for i,params in enumerate(candidates):
            model=fit_candidate(method,params,Z['fit'],seed)
            selection_scores=score(method,model,Z['selection'])
            ap=float(average_precision_score(y_selection,selection_scores))
            trials.append({'method':method,'candidate_order':i,'params':params,'selection_AP':ap})
            if best is None or ap>best['selection_AP']:
                best={'model':model,'params':params,'selection_AP':ap,'scores':selection_scores}
        best['threshold']=choose_threshold(y_selection,best['scores'])
        chosen[method]=best
        joblib.dump(best['model'],out/f'detector_{method}.joblib',compress=3)
    if mse_scores is not None:
        chosen['MSE']={'params':{},'selection_AP':float(average_precision_score(y_selection,mse_scores['selection'])),
                       'threshold':choose_threshold(y_selection,mse_scores['selection'])}

    ref=references[ds]
    cols=['sample_key','engine_key','UnitNumber','Time','RUL','late_life_le_20','late_life_le_30','late_life_le_40']
    held=ref.loc[masks['outer_valid'],cols].reset_index(drop=True)
    selection_frame=ref.loc[masks['selection'],cols].reset_index(drop=True)
    rows=[];pred_frames=[];selection_frames=[]
    for method,best in chosen.items():
        scores=mse_scores['outer_valid'] if method=='MSE' else score(method,best['model'],Z['outer_valid'])
        for w in [20,30,40]:
            y=held[f'late_life_le_{w}'].to_numpy(dtype=int)
            require(len(np.unique(y))==2,'Outer referansı tek sınıf.')
            rows.append({**identity,'method':method,'RUL_window':w,'selected_params':best['params'],
                         'selection_AP_primary':best['selection_AP'],'threshold':best['threshold'],
                         **metrics(y,scores,best['threshold'])})
        pred=held.copy();pred['method']=method;pred['score']=scores;pred['threshold']=best['threshold']
        pred['prediction']=(scores>=best['threshold']).astype(int);pred_frames.append(pred)
        selected=selection_frame.copy();selected['method']=method
        selected['score']=mse_scores['selection'] if method=='MSE' else best['scores']
        selected['threshold']=best['threshold'];selection_frames.append(selected)
    pd.concat(pred_frames,ignore_index=True).to_csv(out/'outer_predictions.csv.gz',index=False,compression='gzip')
    pd.concat(selection_frames,ignore_index=True).to_csv(out/'selection_scores.csv.gz',index=False,compression='gzip')
    pd.DataFrame(rows).to_csv(out/'metrics.csv',index=False)
    write_json(out/'selection_trials.json',trials)
    write_json(out/'representation_info.json',representation_info)
    result={'status':'PASS','plan_sha256':PLAN_SHA256,'identity':identity,'metrics':rows,
            'selection_trials':trials,'representation_info':representation_info,'fold_info':fold_info,
            'test_files_read':False,'seconds':time.perf_counter()-started,
            'output_sha256':{p.name:digest(p) for p in out.iterdir()
                             if p.is_file() and p.name!='result.json' and not p.name.endswith('.tmp')}}
    write_json(out/'result.json',result)
    print('  TAMAMLANDI:',name,'| dakika:',round(result['seconds']/60,2),flush=True)
    return result


In [ ]:
jobs=[]
total=len(BLOCKS)*len(SEEDS)*len(FOLDS)*len(REPRESENTATIONS)
require(total==450,'09 planındaki iş sayısı değişmiş.')
for block in BLOCKS:
    ds=block['dataset'];mode=block['preprocessing']
    for seed in SEEDS:
        for fold in FOLDS:
            prepared=None
            for name in REPRESENTATIONS:
                print(f'[{len(jobs)+1}/{total}] {ds} / {mode} / seed={seed} / fold={fold} / {name}',flush=True)
                cached=cached_job(ds,mode,seed,fold,name)
                if cached is not None:
                    print('  KAYITTAN',flush=True);jobs.append(cached);continue
                if prepared is None:prepared=prepare_fold(ds,mode,fold)
                jobs.append(run_subjob(ds,mode,seed,fold,name,prepared))
            del prepared
            gc.collect()
print('450 temsil alt işi tamamlandı.')


In [ ]:
METRICS=['AP','ROC_AUC','precision','recall','F1','Macro_F1']

def paired_comparison(left,right,description):
    merged=left.merge(right,on=['seed','fold'],suffixes=('_left','_right'),validate='one_to_one')
    require(len(merged)==30,'Eşleştirilmiş karşılaştırmada 30 koşu olmalı.')
    result={**description,'pairs':30,'direction':'left minus right'}
    for metric in ['AP','F1']:
        d=merged[metric+'_left']-merged[metric+'_right']
        result.update({f'delta_{metric}_mean':float(d.mean()),f'delta_{metric}_descriptive_sd':float(d.std(ddof=1)),
                       f'positive_delta_{metric}_runs':int((d>1e-12).sum()),
                       f'negative_delta_{metric}_runs':int((d< -1e-12).sum()),
                       f'tied_delta_{metric}_runs':int((d.abs()<=1e-12).sum())})
    return result

def summarize(jobs):
    wanted={(b['dataset'],b['preprocessing'],s,f,r) for b in BLOCKS for s in SEEDS for f in FOLDS for r in REPRESENTATIONS}
    observed=[tuple(j['identity'][k] for k in ['dataset','preprocessing','seed','fold','representation']) for j in jobs]
    require(len(jobs)==450 and len(set(observed))==450 and set(observed)==wanted,'Tamamlanan iş kapsamı farklı.')
    table=pd.DataFrame([r for j in jobs for r in j['metrics']])
    keys=['dataset','preprocessing','seed','fold','representation','method','RUL_window']
    require(len(table)==4590 and not table.duplicated(keys).any(),'4590 benzersiz metrik satırı bekleniyor.')
    overall=[];per_seed=[]
    groups=['dataset','preprocessing','representation','method','RUL_window']
    require(table.groupby(groups).ngroups==153,'Beklenen 153 sonuç grubu yok.')
    for key,g in table.groupby(groups,sort=True):
        require(len(g)==30 and set(zip(g.seed,g.fold))=={(s,f) for s in SEEDS for f in FOLDS},'Eksik fold/seed.')
        row=dict(zip(groups,key));row['RUL_window']=int(row['RUL_window']);row['runs']=30
        for metric in METRICS:
            row[metric+'_mean']=float(g[metric].mean());row[metric+'_descriptive_sd']=float(g[metric].std(ddof=1))
        overall.append(row)
        for seed,q in g.groupby('seed'):
            entry={k:v for k,v in row.items() if k in groups};entry['seed']=int(seed)
            for metric in METRICS:
                entry[metric+'_fold_mean']=float(q[metric].mean());entry[metric+'_fold_sd']=float(q[metric].std(ddof=1))
            per_seed.append(entry)
    primary=table.loc[table.RUL_window.eq(30)]
    pairs=[]
    contrasts=[(r,'Raw') for r in REPRESENTATIONS if r!='Raw']+[
        ('AE_deep_d4','PCA_d4'),('AE_deep_d4','PCA_scaled_d4'),('AE_deep_d4','AE_deep_d21')]
    for block in BLOCKS:
        ds=block['dataset'];mode=block['preprocessing']
        subset=primary.loc[primary.dataset.eq(ds)&primary.preprocessing.eq(mode)]
        for left,right in contrasts:
            for method in CANDIDATES:
                l=subset.loc[subset.representation.eq(left)&subset.method.eq(method)]
                r=subset.loc[subset.representation.eq(right)&subset.method.eq(method)]
                pairs.append(paired_comparison(l,r,{'dataset':ds,'preprocessing':mode,'method':method,'left':left,'right':right}))
    condition_pairs=[]
    fd2=primary.loc[primary.dataset.eq('FD002')]
    for (rep,method),g in fd2.groupby(['representation','method']):
        condition_pairs.append(paired_comparison(g.loc[g.preprocessing.eq('conditionwise')],g.loc[g.preprocessing.eq('global')],
                               {'dataset':'FD002','representation':rep,'method':method,'left':'conditionwise','right':'global'}))
    ae=[{**j['identity'],**j['representation_info']} for j in jobs if j['identity']['representation'].startswith('AE_')]
    fold_info=[]
    for j in jobs:
        if j['identity']['representation']=='Raw':fold_info.append({**j['identity'],**j['fold_info']})
    table.to_csv(RUN_DIR/'all_fold_metrics.csv',index=False)
    pd.DataFrame(overall).to_csv(RUN_DIR/'aggregate_metrics.csv',index=False)
    pd.DataFrame(per_seed).to_csv(RUN_DIR/'per_seed_metrics.csv',index=False)
    pd.DataFrame(pairs).to_csv(RUN_DIR/'paired_representation_differences.csv',index=False)
    pd.DataFrame(condition_pairs).to_csv(RUN_DIR/'paired_preprocessing_differences.csv',index=False)
    summary={'stage':'10_additional_grouped_cv','status':'PASS','protocol_sha256':EXPECTED_PROTOCOL,
             'plan_sha256':PLAN_SHA256,'versions':VERSIONS,'completed_fold_seed_blocks':90,
             'completed_representation_jobs':len(jobs),'AE_fits':len(ae),'analysis_blocks':BLOCKS,
             'folds':list(FOLDS),'seeds':list(SEEDS),'representations':REPRESENTATIONS,'detectors':list(CANDIDATES),
             'test_files_read':False,'RUL_in_model_inputs':False,'statistical_tests_performed':False,
             'reference':'RUL-derived late-life proxy; primary RUL<=30; fixed-score/threshold sensitivity 20,40',
             'selection':'AP then positive F1 threshold on selection engines only; no refit',
             'aggregation':'equal-weight fold/seed means within dataset/preprocessing; SD descriptive; no pooled AP',
             'scope':'IF/LOF/kNN and two standalone AE-MSE scores; not all reviewer requests',
             'prior_information':protocol['prior_information'],
             'primary_results':[r for r in overall if r['RUL_window']==30],
             'sensitivity_results':[r for r in overall if r['RUL_window']!=30],
             'per_seed_results':per_seed,'paired_representations':pairs,'paired_FD002_preprocessing':condition_pairs,
             'AE_training':ae,'fold_preprocessing':fold_info,
             'fold_primary_results':primary.to_dict(orient='records'),
             'output_file_sha256':{name:digest(RUN_DIR/name) for name in ['run_plan.json','all_fold_metrics.csv',
                 'aggregate_metrics.csv','per_seed_metrics.csv','paired_representation_differences.csv','paired_preprocessing_differences.csv']}}
    require(len(summary['primary_results'])==51 and len(summary['sensitivity_results'])==102 and len(ae)==180,
            'Özet kapsamı farklı.')
    write_json(RUN_DIR/'additional_cv_summary.json',summary)
    return summary

additional_cv_summary=summarize(jobs)
print(pd.DataFrame(additional_cv_summary['primary_results'])[
    ['dataset','preprocessing','representation','method','AP_mean','F1_mean']].to_string(index=False))
print('EK DOĞRULAMA: PASS — deneyler tamamlandı; üstünlük iddiası değildir.')
print('Gönderilecek dosya:',(RUN_DIR/'additional_cv_summary.json').resolve())
